# Limpieza e integración del dataset Bechdel Test

## Importación del dataset de TidyTuesday

In [ ]:
import pandas as pd
import requests
import time
import re

# --- Bechdel / TidyTuesday dataset ---
movies = pd.read_csv("movies.csv")
movies["imdb_id"] = movies["imdb_id"].astype(str)

# --- IMDb data from GitHub (no local file needed) ---
url_imdb = "https://raw.githubusercontent.com/brianckeegan/Bechdel/master/imdb_data.json"
resp = requests.get(url_imdb)
resp.raise_for_status()
data = resp.json()

imdb = pd.json_normalize(data)
imdb = imdb.rename(columns={"imdbID": "imdb_id"})
imdb["imdb_id"] = imdb["imdb_id"].astype(str)

print(imdb.columns)  # just to see what's available

Index(['Plot', 'Rated', 'Response', 'Language', 'Title', 'Country', 'Writer',
       'Metascore', 'imdbRating', 'Director', 'Released', 'Actors', 'Year',
       'Genre', 'Awards', 'Runtime', 'Type', 'Poster', 'imdbVotes', 'imdb_id',
       'Error'],
      dtype='object')


## Limpieza básica

In [ ]:
# 1. Renombrar columnas de IMDb a minúsculas
imdb_renamed = imdb.rename(columns={
    "Director": "director",
    "Writer": "writer",
    "Genre": "genre"
})

# 2. Hacer el merge solo con las columnas que nos interesan
merged = movies.merge(
    imdb_renamed[["imdb_id", "director", "writer", "genre"]],
    on="imdb_id",
    how="left",
    suffixes=("", "_imdb")   # ahora sí se aplican a director/writer/genre
)

# 3. Rellenar los NA de movies con la info de IMDb
for col in ["director", "writer", "genre"]:
    merged[col] = merged[col].fillna(merged[f"{col}_imdb"])

# 4. Eliminar las columnas auxiliares *_imdb
cols_to_drop = [c for c in merged.columns if c.endswith("_imdb")]
merged = merged.drop(columns=cols_to_drop)

merged.head()

,year,imdb,title,test,clean_test,binary,budget,domgross,intgross,code,...,director,released,actors,genre,awards,runtime,type,poster,imdb_votes,error
0,2013,tt1711425,21 &amp; Over,notalk,notalk,FAIL,13000000,25682380.0,42195766.0,2013FAIL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,tt1343727,Dredd 3D,ok-disagree,ok,PASS,45000000,13414714.0,40868994.0,2012PASS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,tt2024544,12 Years a Slave,notalk-disagree,notalk,FAIL,20000000,53107035.0,158607035.0,2013FAIL,...,Steve McQueen,08 Nov 2013,"Chiwetel Ejiofor, Dwight Henry, Dickie Gravois...","Biography, Drama, History",Won 3 Oscars. Another 131 wins & 137 nominations.,134 min,movie,http://ia.media-imdb.com/images/M/MV5BMjExMTEz...,"143,446",NaN
3,2013,tt1272878,2 Guns,notalk,notalk,FAIL,61000000,75612460.0,132493015.0,2013FAIL,...,Baltasar Kormákur,02 Aug 2013,"Denzel Washington, Mark Wahlberg, Paula Patton...","Action, Comedy, Crime",1 win.,109 min,movie,http://ia.media-imdb.com/images/M/MV5BNTQ5MTgz...,"87,301",NaN
4,2013,tt0453562,42,men,men,FAIL,40000000,95020213.0,95020213.0,2013FAIL,...,Brian Helgeland,12 Apr 2013,"Chadwick Boseman, Harrison Ford, Nicole Behari...","Biography, Drama, Sport",3 wins & 13 nominations.,128 min,movie,http://ia.media-imdb.com/images/M/MV5BMTQwMDU4...,"43,608",NaN


## Creación de variables derivadas

En este apartado se infiere el género del equipo creativo a partir de los nombres de pila.

In [ ]:
# Directores en formato largo
directors_long = (
    merged[["imdb_id", "title", "year", "director"]]
    .assign(director=lambda df: df["director"].fillna(""))
    .assign(director_list=lambda df: df["director"].str.split(","))
    .explode("director_list")
)

directors_long["director"] = directors_long["director_list"].str.strip()
directors_long = directors_long.drop(columns=["director_list"])
directors_long = directors_long[directors_long["director"] != ""]

# Escritores en formato largo
writers_long = (
    merged[["imdb_id", "title", "year", "writer"]]
    .assign(writer=lambda df: df["writer"].fillna(""))
    .assign(writer_list=lambda df: df["writer"].str.split(","))
    .explode("writer_list")
)

writers_long["writer"] = writers_long["writer_list"].str.strip()
writers_long = writers_long.drop(columns=["writer_list"])
writers_long = writers_long[writers_long["writer"] != ""]


In [ ]:
def clean_firstname(full_name):
    """
    Coge un nombre completo y retornaTake a full name and return a cleaned first name suitable for gender APIs.
    """
    if pd.isna(full_name) or full_name == "":
        return None

    # Pasar a minúscula y eliminar caracteres especiales excepto guiones y espacios
    name = re.sub(r"[^A-Za-zÀ-ÿ\- ]", "", full_name).strip()
    if not name:
        return None

    # Coger el primer token como nombre de pila
    first = name.split()[0]
    if len(first) <= 1:
        return None
    return first

directors_long["director_firstname"] = directors_long["director"].apply(clean_firstname)
writers_long["writer_firstname"] = writers_long["writer"].apply(clean_firstname)

directors_long = directors_long.dropna(subset=["director_firstname"])
writers_long = writers_long.dropna(subset=["writer_firstname"])

In [ ]:
director_names = directors_long["director_firstname"].unique()
writer_names = writers_long["writer_firstname"].unique()

all_firstnames = pd.Series(list(set(director_names) | set(writer_names)))
all_firstnames = all_firstnames[all_firstnames.notna() & (all_firstnames != "")]
print(len(all_firstnames), "unique first names")

1017 unique first names


In [ ]:
pip install gender-guesser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.3/379.3 kB 12.2 MB/s eta 0:00:00


In [ ]:
import gender_guesser.detector as gender
import pandas as pd

d = gender.Detector(case_sensitive=False)

def guess_gender(name):
    """
    Usa gender-guesser para predecir género por su nombre de pila.
    Mapea a 'male' / 'female' / None.
    """
    if pd.isna(name) or name == "":
        return None

    g = d.get_gender(name)
    # Posibles outputs: 'male', 'female', 'mostly_male', 'mostly_female', 'andy', 'unknown'
    if g in ["male", "mostly_male"]:
        return "male"
    elif g in ["female", "mostly_female"]:
        return "female"
    else:
        return None

In [ ]:
# Directores
directors_long["director_gender"] = directors_long["director_firstname"].apply(guess_gender)

# Guionistas
writers_long["writer_gender"] = writers_long["writer_firstname"].apply(guess_gender)


In [ ]:
# Directores: contar males/females por imdb_id
directors_summary = (
    directors_long
    .groupby(["imdb_id", "director_gender"])
    .size()
    .unstack(fill_value=0)
    .add_prefix("directors_")   # directors_female, directors_male
    .reset_index()
)

# Guionistas
writers_summary = (
    writers_long
    .groupby(["imdb_id", "writer_gender"])
    .size()
    .unstack(fill_value=0)
    .add_prefix("writers_")
    .reset_index()
)


In [ ]:
movies_enriched = (
    merged
    .merge(directors_summary, on="imdb_id", how="left")
    .merge(writers_summary,   on="imdb_id", how="left")
)

count_cols = [c for c in movies_enriched.columns
              if c.startswith("directors_") or c.startswith("writers_")]
movies_enriched[count_cols] = movies_enriched[count_cols].fillna(0).astype(int)

# Creamos también variables categóricas que indican si hay mujeres en el equipo
movies_enriched["has_female_director"] = movies_enriched.get("directors_female", 0) > 0
movies_enriched["has_female_writer"]   = movies_enriched.get("writers_female", 0) > 0

movies_enriched.head(20)

,year,imdb,title,test,clean_test,binary,budget,domgross,intgross,code,...,type,poster,imdb_votes,error,directors_female,directors_male,writers_female,writers_male,has_female_director,has_female_writer
0,2013,tt1711425,21 &amp; Over,notalk,notalk,FAIL,13000000,25682380.0,42195766.0,2013FAIL,...,NaN,NaN,NaN,NaN,0,0,0,0,False,False
1,2012,tt1343727,Dredd 3D,ok-disagree,ok,PASS,45000000,13414714.0,40868994.0,2012PASS,...,NaN,NaN,NaN,NaN,0,0,0,0,False,False
2,2013,tt2024544,12 Years a Slave,notalk-disagree,notalk,FAIL,20000000,53107035.0,158607035.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BMjExMTEz...,"143,446",NaN,0,1,0,2,False,False
3,2013,tt1272878,2 Guns,notalk,notalk,FAIL,61000000,75612460.0,132493015.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BNTQ5MTgz...,"87,301",NaN,0,1,0,2,False,False
4,2013,tt0453562,42,men,men,FAIL,40000000,95020213.0,95020213.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BMTQwMDU4...,"43,608",NaN,0,1,0,1,False,False
5,2013,tt1335975,47 Ronin,men,men,FAIL,225000000,38362475.0,145803842.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BMTc0MjE2...,"25,735",NaN,0,1,0,4,False,False
6,2013,tt1606378,A Good Day to Die Hard,notalk,notalk,FAIL,92000000,67349198.0,304249198.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BMTcwNzgy...,"123,837",NaN,0,1,0,2,False,False
7,2013,tt2194499,About Time,ok-disagree,ok,PASS,12000000,15323921.0,87324746.0,2013PASS,...,movie,http://ia.media-imdb.com/images/M/MV5BMTA1ODUz...,"85,871",NaN,0,1,0,1,False,False
8,2013,tt1814621,Admission,ok,ok,PASS,13000000,18007317.0,18007317.0,2013PASS,...,movie,http://ia.media-imdb.com/images/M/MV5BOTE2OTkw...,"18,973",NaN,0,1,1,1,False,True
9,2013,tt1815862,After Earth,notalk,notalk,FAIL,130000000,60522097.0,244373198.0,2013FAIL,...,movie,http://ia.media-imdb.com/images/M/MV5BMTY3MzQy...,"108,264",NaN,0,0,0,2,False,False


## Integración del dataset Hollywood Age Gap

Añadimos datos del dataset Hollywood Age Gap para incluir un análisis secundario de la diferencia de edad entre las parejas en pantalla y sus géneros.
De este dataset se obtiene quién es mayor y quien es menor en la pareja, si un hombre o una mujer, y la diferencia de edad entre ambos.

In [ ]:
import re

def normalize_title(s):
    if pd.isna(s):
        return None
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

movies_enriched["title_norm"] = movies_enriched["title"].apply(normalize_title)
movies_enriched["year_int"]   = movies_enriched["year"].astype("Int64")

In [ ]:
agegap = pd.read_csv("age_gaps.csv")

agegap_main = agegap[agegap["couple_number"] == 1].copy()

# Mantenemos columnas de interés
agegap_small = agegap[[
    "movie_name",
    "release_year",
    "character_1_gender",
    "character_2_gender",
    "actor_1_age",
    "actor_2_age"
]].copy()

agegap_small["title_norm"] = agegap_small["movie_name"].apply(normalize_title)
agegap_small["year_int"]   = agegap_small["release_year"].astype("Int64")

agegap_unique = (
    agegap_small
    .sort_values(["title_norm", "year_int"])
    .drop_duplicates(subset=["title_norm", "year_int"], keep="first")
)


In [ ]:
def normalize_title(s):
    if pd.isna(s):
        return None
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# En movies_enriched
movies_enriched["title_norm"] = movies_enriched["title"].apply(normalize_title)
movies_enriched["year_int"]   = movies_enriched["year"].astype("Int64")

# En agegap
agegap_small["title_norm"] = agegap_small["movie_name"].apply(normalize_title)
agegap_small["year_int"]   = agegap_small["release_year"].astype("Int64")

movies_with_agegap = movies_enriched.merge(
    agegap_small[[
        "title_norm", "year_int",
        "character_1_gender", "character_2_gender",
        "actor_1_age", "actor_2_age"
    ]],
    on=["title_norm", "year_int"],
    how="left"
)


In [ ]:
movies_with_agegap = movies_with_agegap.rename(columns={
    "character_1_gender": "older_char_gender",
    "character_2_gender": "younger_char_gender",
    "actor_1_age": "older_actor_age",
    "actor_2_age": "younger_actor_age"
})


In [ ]:
print(len(movies_with_agegap))
movies_with_agegap.head(10)

1904


,year,imdb,title,test,clean_test,binary,budget,domgross,intgross,code,...,writers_female,writers_male,has_female_director,has_female_writer,title_norm,year_int,older_char_gender,younger_char_gender,older_actor_age,younger_actor_age
0,2013,tt1711425,21 &amp; Over,notalk,notalk,FAIL,13000000,25682380.0,42195766.0,2013FAIL,...,0,0,False,False,21 amp over,2013,NaN,NaN,NaN,NaN
1,2012,tt1343727,Dredd 3D,ok-disagree,ok,PASS,45000000,13414714.0,40868994.0,2012PASS,...,0,0,False,False,dredd 3d,2012,NaN,NaN,NaN,NaN
2,2013,tt2024544,12 Years a Slave,notalk-disagree,notalk,FAIL,20000000,53107035.0,158607035.0,2013FAIL,...,0,2,False,False,12 years a slave,2013,NaN,NaN,NaN,NaN
3,2013,tt1272878,2 Guns,notalk,notalk,FAIL,61000000,75612460.0,132493015.0,2013FAIL,...,0,2,False,False,2 guns,2013,NaN,NaN,NaN,NaN
4,2013,tt0453562,42,men,men,FAIL,40000000,95020213.0,95020213.0,2013FAIL,...,0,1,False,False,42,2013,man,woman,37.0,28.0
5,2013,tt1335975,47 Ronin,men,men,FAIL,225000000,38362475.0,145803842.0,2013FAIL,...,0,4,False,False,47 ronin,2013,man,woman,49.0,32.0
6,2013,tt1606378,A Good Day to Die Hard,notalk,notalk,FAIL,92000000,67349198.0,304249198.0,2013FAIL,...,0,2,False,False,a good day to die hard,2013,NaN,NaN,NaN,NaN
7,2013,tt2194499,About Time,ok-disagree,ok,PASS,12000000,15323921.0,87324746.0,2013PASS,...,0,1,False,False,about time,2013,woman,man,35.0,30.0
8,2013,tt1814621,Admission,ok,ok,PASS,13000000,18007317.0,18007317.0,2013PASS,...,1,1,False,True,admission,2013,NaN,NaN,NaN,NaN
9,2013,tt1815862,After Earth,notalk,notalk,FAIL,130000000,60522097.0,244373198.0,2013FAIL,...,0,2,False,False,after earth,2013,NaN,NaN,NaN,NaN


In [ ]:
def who_is_older(row):
    """
    Retorna quién es mayor en la pareja.

    """
    g_old = str(row["older_char_gender"]).lower() if pd.notna(row["older_char_gender"]) else None
    g_yng = str(row["younger_char_gender"]).lower() if pd.notna(row["younger_char_gender"]) else None

    if g_old in ("man", "male") and g_yng in ("woman", "female"):
        return "man"
    elif g_old in ("woman", "female") and g_yng in ("man", "male"):
        return "woman"
    elif g_old is None or g_yng is None:
        return None
    else:
        # Mismo género o valores inesperados
        return "same_gender"

movies_with_agegap["older_person"] = movies_with_agegap.apply(who_is_older, axis=1)


In [ ]:
import numpy as np
import pandas as pd

movies_with_agegap["age_gap"] = np.where(
    movies_with_agegap[["older_actor_age", "younger_actor_age"]].notna().all(axis=1),
    (movies_with_agegap["older_actor_age"] - movies_with_agegap["younger_actor_age"]).abs(),
    np.nan
)

In [ ]:
movies_with_agegap = movies_with_agegap.drop(columns=[
    "title_norm",
    "year_int"
])

In [ ]:
movies_with_agegap.to_excel("movies_with_agegap.xlsx", index=False)